In [1]:
import pandas as pd

df = pd.read_csv("swiggy_customer_data.csv")

df.head()

,customer_id,age,city,orders_last_30_days,avg_order_value,days_since_last_order,coupon_used_last_month,customer_support_calls,churn
0,10001,56,Bangalore,19,1472,36,1,9,0
1,10002,46,Mumbai,3,1081,46,0,5,1
2,10003,32,Bangalore,9,1020,75,1,6,0
3,10004,25,Chennai,2,821,36,1,2,0
4,10005,38,Bangalore,8,1284,48,0,6,0


In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["city"] = le.fit_transform(df["city"])
df.head()

,customer_id,age,city,orders_last_30_days,avg_order_value,days_since_last_order,coupon_used_last_month,customer_support_calls,churn
0,10001,56,0,19,1472,36,1,9,0
1,10002,46,3,3,1081,46,0,5,1
2,10003,32,0,9,1020,75,1,6,0
3,10004,25,1,2,821,36,1,2,0
4,10005,38,0,8,1284,48,0,6,0


In [3]:
X = df.drop("churn", axis=1)
y = df["churn"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (500, 8)
y Shape: (500,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (350, 8)
Testing Data: (150, 8)


In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [7]:
y_pred = model.predict(X_test)

y_pred[:10]

array([0, 1, 1, 0, 0, 0, 0, 0, 0, 0], dtype=int64)

In [8]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.98

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       129
           1       1.00      0.86      0.92        21

    accuracy                           0.98       150
   macro avg       0.99      0.93      0.96       150
weighted avg       0.98      0.98      0.98       150


Confusion Matrix:
[[129   0]
 [  3  18]]


In [9]:
df["risk_category"] = df["churn"].apply(
    lambda x: "High Risk" if x == 1 else "Low Risk"
)
df[["customer_id", "churn", "risk_category"]].head()

,customer_id,churn,risk_category
0,10001,0,Low Risk
1,10002,1,High Risk
2,10003,0,Low Risk
3,10004,0,Low Risk
4,10005,0,Low Risk


In [10]:
def customer_action(risk):
    
    if risk == "High Risk":
        return "20% Discount Coupon"
    
    else:
        return "Thank You Message"

df["action"] = df["risk_category"].apply(customer_action)

df[["customer_id","risk_category","action"]].head()

,customer_id,risk_category,action
0,10001,Low Risk,Thank You Message
1,10002,High Risk,20% Discount Coupon
2,10003,Low Risk,Thank You Message
3,10004,Low Risk,Thank You Message
4,10005,Low Risk,Thank You Message


In [11]:
def generate_message(row):
    
    if row["risk_category"] == "High Risk":
        
        return (
            f"Hi Customer {row['customer_id']}, "
            f"We miss you! Enjoy a 20% discount coupon "
            f"on your next Swiggy order."
        )
    
    else:
        
        return (
            f"Hi Customer {row['customer_id']}, "
            f"Thank you for being a valued Swiggy customer."
        )

df["message"] = df.apply(generate_message, axis=1)

df[["customer_id","risk_category","message"]].head()

,customer_id,risk_category,message
0,10001,Low Risk,"Hi Customer 10001, Thank you for being a value..."
1,10002,High Risk,"Hi Customer 10002, We miss you! Enjoy a 20% di..."
2,10003,Low Risk,"Hi Customer 10003, Thank you for being a value..."
3,10004,Low Risk,"Hi Customer 10004, Thank you for being a value..."
4,10005,Low Risk,"Hi Customer 10005, Thank you for being a value..."


In [12]:
df.to_csv(
    "customer_campaign_output.csv",
    index=False
)

print("Campaign File Generated")

Campaign File Generated
